# Day 4 — Structural Cascade Stress Analysis

## Objective and scope

This notebook evaluates the existing `sim.simulate.simulate_cascade` abstraction using the frozen core network and the agreed 14-day outgoing-departure baseline. It is a **structural stress analysis**, not operational OD rerouting, a warehouse-capacity estimate, or a queue simulation.

The clean execution path contains toy validation, synthetic sensitivity, input-lineage assertions, construction of the real-data baseline, and one failure scenario per core hub. Historical and superseded analyses are preserved separately and are not executed here.

In [1]:
from pathlib import Path
import hashlib, importlib.metadata, json, platform, sys
from datetime import datetime, timezone
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT=Path.cwd().resolve()
while not (ROOT/"data/processed/split_trip_ids.csv").is_file():
    if ROOT.parent==ROOT: raise FileNotFoundError("Could not locate SupplyShield project root")
    ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from sim.simulate import build_neighbors, simulate_cascade
DATA=ROOT/"data/processed"
DAY4_ROOT=ROOT/"experiments/day4"
NOTEBOOK_PATH="notebooks/B04_cascade_day4_final_v3.ipynb"
def next_output_dir(parent):
    for i in range(1,1000):
        p=parent/f"final_v{i}"
        if not p.exists(): return p
    raise RuntimeError("No unused versioned Day 4 output directory")
OUT_DIR=next_output_dir(DAY4_ROOT)
print("Python",platform.python_version(),"pandas",pd.__version__,"NumPy",np.__version__,"Matplotlib",matplotlib.__version__)
print("Output directory selected (must not already exist):",OUT_DIR.relative_to(ROOT))

Python 3.14.2 pandas 2.3.3 NumPy 2.4.3 Matplotlib 3.10.8
Output directory selected (must not already exist): experiments\day4\final_v2


## Structural cascade model: methodology and assumptions

`build_neighbors` symmetrizes the directed lane list into an undirected neighbor graph. For each failure round, `simulate_cascade` redistributes the failed hub’s entire accumulated scalar load among alive neighbors, weighted by `max(capacity - current_load, 0)`. When all spare-room weights are zero, it falls back to capacity-proportional weights. A hub fails when updated load exceeds `tolerance × capacity`.

The abstraction does not preserve OD flows, direction, route choice, or only-excess rerouting. Dispatch estimates are used as a structural proxy for spare room; receiving capacity is not measured. It does not represent the three zero-dispatch destination-only hubs as final-delivery sinks.

## Toy simulator validation — synthetic demonstration

The hand-specified eight-hub cases validate conservation, propagation, and tolerance behavior. These results are synthetic and do not describe the observed core network.

In [2]:
toy_edges=[("H1","H2"),("H1","H3"),("H2","H4"),("H3","H4"),("H4","H5"),("H5","H6"),("H5","H7"),("H6","H8"),("H7","H8")]
toy_cap={"H1":100,"H2":100,"H3":100,"H4":150,"H5":120,"H6":100,"H7":100,"H8":100}
toy_load={"H1":60,"H2":70,"H3":70,"H4":100,"H5":80,"H6":60,"H7":60,"H8":50}
toy_nb=build_neighbors(toy_edges)
r=simulate_cascade(toy_cap,toy_load,toy_nb,["H4"])
assert r["n_failed"]==1 and r["unserved_load"]==0 and r["converged"]
assert all(abs(r["final_load"][h]-x)<1e-6 for h,x in {"H2":100,"H3":100,"H5":120}.items())
toy_stress={**toy_load,"H4":120}
r=simulate_cascade(toy_cap,toy_stress,toy_nb,["H4"])
assert r["n_failed"]==8 and r["n_rounds"]==4 and abs(r["unserved_load"]-570)<1e-6
r=simulate_cascade(toy_cap,toy_stress,toy_nb,["H4"],tolerance=1.1)
assert r["n_failed"]==1 and r["unserved_load"]==0
assert all(simulate_cascade(toy_cap,toy_stress,toy_nb,[h])["conserved"] for h in toy_cap)
print("PASS: calm case, full cascade case, tolerance case, and conservation checks")

PASS: calm case, full cascade case, tolerance case, and conservation checks


## Core network inspection

Load the frozen nodes and directed lanes. The analysis checks directed lane-key integrity before creating the undirected structural neighbor graph.

In [3]:
core_nodes=pd.read_csv(DATA/"core_nodes.csv")
core_edges=pd.read_csv(DATA/"core_edges.csv")
assert core_nodes.hub.notna().all(),"Core node IDs contain missing values"
assert core_nodes.hub.is_unique,"Core node IDs must be unique"
assert core_edges[["source_center","destination_center"]].notna().all().all(),"Directed core lane keys contain missing values"
assert not core_edges.duplicated(["source_center","destination_center"]).any(),"Directed core lane keys must be unique"
assert len(core_nodes)==100 and len(core_edges)==292,"Frozen network size differs from expected"
core_hubs=core_nodes.hub.tolist()
edge_pairs=set(zip(core_edges.source_center,core_edges.destination_center))
assert set(core_edges.source_center)|set(core_edges.destination_center) <= set(core_hubs),"Edge endpoint absent from core nodes"
assert core_edges.source.equals(core_edges.source_center) and core_edges.destination.equals(core_edges.destination_center),"Network aliases differ"
neighbors=build_neighbors(list(edge_pairs))
print("Core hubs:",len(core_hubs),"| frozen directed lanes:",len(edge_pairs),"| zero dispatch-capacity hubs:",int((core_nodes.capacity_est==0).sum()))

Core hubs: 100 | frozen directed lanes: 292 | zero dispatch-capacity hubs: 3


## Synthetic stress experiments

Uniform fractions of dispatch proxies are used as synthetic load scenarios. They are sensitivity demonstrations only. The one-step local spare-room diagnostic is compared with fresh simulator results dynamically; equality is neither expected nor asserted because a one-step screen is not the full iterative process.

In [4]:
capacity=core_nodes.set_index("hub").capacity_est.astype(float).to_dict()
stress_rows=[]
for stress in np.round(np.arange(.50,1.001,.05),2):
    synthetic_load={h:stress*c for h,c in capacity.items()}
    for tolerance in [1.0,1.1,1.2,1.3]:
        rs=[simulate_cascade(capacity,synthetic_load,neighbors,[h],tolerance=tolerance,max_rounds=30) for h in core_hubs]
        stress_rows.append({"stress_fraction":float(stress),"tolerance":tolerance,"initial_failures_tested":len(rs),
            "secondary_cascade_count":sum(x["n_failed"]>1 for x in rs),"cascade_share":float(np.mean([x["n_failed"]>1 for x in rs])),
            "mean_hubs_failed":float(np.mean([x["n_failed"] for x in rs])),"largest_cascade_hubs":int(max(x["n_failed"] for x in rs)),
            "mean_unserved_share":float(np.mean([x["unserved_share"] for x in rs])),"non_converged_count":sum(not x["converged"] for x in rs)})
synthetic_stress=pd.DataFrame(stress_rows)
assert len(synthetic_stress)==44
fig,ax=plt.subplots(figsize=(7,4.5))
for tol,g in synthetic_stress.groupby("tolerance"):
    ax.plot(g.stress_fraction,g.cascade_share,marker="o",label=f"tolerance={tol:.1f}")
ax.set(xlabel="Uniform synthetic load / dispatch proxy",ylabel="Share of single-hub failures causing secondary cascades",title="Synthetic structural sensitivity (not observed traffic)")
ax.grid(True,alpha=.3); ax.legend(); fig.tight_layout()
stress90={h:.9*c for h,c in capacity.items()}
stress90_sim={h:simulate_cascade(capacity,stress90,neighbors,[h]) for h in core_hubs}
trigger=[]
for h in core_hubs:
    spare=sum(max(capacity[j]-stress90[j],0.0) for j in neighbors.get(h,[]))
    trigger.append({"hub":h,"neighbor_spare":spare,"one_step_trigger":bool(spare>0 and stress90[h]>spare),
                    "full_simulator_secondary_cascade":bool(stress90_sim[h]["n_failed"]>1)})
trigger_diagnostic=pd.DataFrame(trigger)
print("Synthetic sweep rows:",len(synthetic_stress))
print("90% scenario: one-step triggers",int(trigger_diagnostic.one_step_trigger.sum()),"; full iterative secondary cascades",int(trigger_diagnostic.full_simulator_secondary_cascade.sum()))
print("These are distinct diagnostics; no hardcoded simulator count or equivalence claim is used.")

Synthetic sweep rows: 44
90% scenario: one-step triggers 39 ; full iterative secondary cascades 38
These are distinct diagnostics; no hardcoded simulator count or equivalence claim is used.


## Real-data inputs and lineage validation

Validate the complete 100-hub × 14-date series against an independent aggregation of `core_train_legs.csv`, using only September 12–25 departures on the frozen directed lanes. No unexpected source-only rows are dropped.

In [5]:
hub_daily=pd.read_csv(DATA/"hub_daily_departures.csv",dtype={"date":str})
core_train=pd.read_csv(DATA/"core_train_legs.csv",parse_dates=["od_start_time","od_end_time","trip_creation_time"])
split_ids=pd.read_csv(DATA/"split_trip_ids.csv")
assert split_ids.trip_uuid.notna().all() and split_ids.trip_uuid.is_unique,"Split IDs must be present and unique"
assert set(split_ids.split.unique())=={"train","calibration","test"},"Unexpected split labels"
split_map=split_ids.set_index("trip_uuid").split
piles=core_train.trip_uuid.map(split_map)
assert piles.notna().all() and set(piles.unique())=={"train"},"Core legs must map exclusively to master training trips"
assert core_train.trip_uuid.notna().all(),"Core training trip IDs are missing"
assert core_train[["source_center","destination_center"]].notna().all().all(),"Core training lane keys are missing"
expected_dates=set(pd.date_range("2018-09-12","2018-09-25",freq="D").strftime("%Y-%m-%d"))
expected_hubs=set(core_nodes.hub)
assert hub_daily[["hub","date","outgoing_legs"]].notna().all().all(),"Daily series has missing values"
vals=hub_daily.outgoing_legs.to_numpy(dtype=float)
assert np.isfinite(vals).all() and (vals>=0).all(),"Daily counts must be finite and nonnegative"
assert not hub_daily.duplicated(["hub","date"]).any(),"Daily hub/date keys must be unique"
assert set(hub_daily.hub)==expected_hubs and set(hub_daily.date)==expected_dates,"Daily series hubs/dates differ from the specification"
expected_grid={(h,d) for h in expected_hubs for d in expected_dates}
assert set(zip(hub_daily.hub,hub_daily.date))==expected_grid and len(hub_daily)==1400,"Expected full 100×14 hub/date grid"
start_date=core_train.od_start_time.dt.strftime("%Y-%m-%d")
mask=start_date.isin(expected_dates)
in_window=core_train.loc[mask].copy()
outside_count=int((~mask).sum())
pairs=set(zip(in_window.source_center,in_window.destination_center))
assert pairs<=edge_pairs,"In-window records include non-core directed lanes"
raw=(in_window.assign(date=start_date.loc[mask].to_numpy()).groupby(["source_center","date"]).size().rename("raw_count").reset_index().rename(columns={"source_center":"hub"}))
assert set(zip(raw.hub,raw.date))<=expected_grid,"Raw aggregation contains unexpected hubs or dates"
comparison=hub_daily.merge(raw,on=["hub","date"],how="outer",validate="one_to_one",indicator=True)
assert not comparison._merge.eq("right_only").any(),"Raw-data-only hub/date combinations found"
left_only=comparison._merge.eq("left_only")
assert (comparison.loc[left_only,"outgoing_legs"]==0).all(),"Supplied-only hub/date combinations are not zero-departure days"
comparison[["outgoing_legs","raw_count"]]=comparison[["outgoing_legs","raw_count"]].fillna(0)
assert (comparison.outgoing_legs==comparison.raw_count).all(),"Independent daily aggregation differs from supplied series"
assert outside_count==38 and len(core_train)==5395 and core_train.trip_uuid.nunique()==4709,"Core training population changed"
assert len(in_window)==5357 and int(hub_daily.outgoing_legs.sum())==5357,"In-window departure total changed"
assert int((hub_daily.outgoing_legs==0).sum())==267,"Zero-departure hub-days changed"
print("PASS split IDs unique; core legs all master train:",piles.value_counts().to_dict())
print("PASS core rows/trips:",len(core_train),core_train.trip_uuid.nunique(),"outside window:",outside_count)
print("PASS grid:",hub_daily.shape,"zero days:",int((hub_daily.outgoing_legs==0).sum()),"departures:",int(hub_daily.outgoing_legs.sum()),"raw comparison rows:",len(comparison))

PASS split IDs unique; core legs all master train: {'train': 5395}
PASS core rows/trips: 5395 4709 outside window: 38
PASS grid: (1400, 3) zero days: 267 departures: 5357 raw comparison rows: 1400


## Real-data baseline and dispatch-capacity proxy

Baseline load is mean daily outgoing departures per hub over all 14 dates, including zero counts. Capacity is the ceiling of the linear 95th percentile of those same daily counts. These are legs per day and a dispatch-throughput proxy, not receiving capacity.

In [6]:
baseline=hub_daily.groupby("hub").outgoing_legs.mean().reindex(core_nodes.hub).astype(float)
p95=hub_daily.groupby("hub").outgoing_legs.quantile(.95,interpolation="linear").reindex(core_nodes.hub)
expected_capacity=np.ceil(p95).astype(int)
node_index=core_nodes.set_index("hub")
assert np.allclose(node_index.loc[baseline.index,"legs_per_day_p95"].to_numpy(float),p95.to_numpy(float),rtol=0,atol=1e-12),"Stored p95 differs from linear daily p95"
assert np.array_equal(node_index.loc[baseline.index,"capacity_est"].to_numpy(int),expected_capacity.to_numpy(int)),"Stored capacity is not ceil(linear p95)"
load=baseline.to_dict(); capacity=node_index.capacity_est.astype(float).to_dict()
assert set(load)==set(capacity)==expected_hubs and baseline.notna().all()
assert all(load[h]<=capacity[h]+1e-9 for h in expected_hubs),"Initial mean baseline exceeds capacity"
baseline_table=pd.DataFrame({"hub":baseline.index,"mean_daily_outgoing_departures":baseline.to_numpy(),"linear_p95_daily_outgoing_departures":p95.to_numpy(),"capacity_est_ceil_p95":expected_capacity.to_numpy()})
print("Baseline: 2018-09-12 to 2018-09-25 inclusive; denominator 14 calendar days per hub")
print("Network daily departures:",float(hub_daily.outgoing_legs.sum())/14,"mean legs/day (fractional allowed)")
print("Capacity recipe reproduced for",len(baseline_table),"hubs")

Baseline: 2018-09-12 to 2018-09-25 inclusive; denominator 14 calendar days per hub
Network daily departures: 382.64285714285717 mean legs/day (fractional allowed)
Capacity recipe reproduced for 100 hubs


## Real-data structural cascade analysis

Fail each hub once, independently, from the same real-data mean-load baseline. Results are conditional on the undirected graph, scalar redistribution rule, dispatch proxy and tolerance 1.0.

In [7]:
real_rows=[]
for hub in core_hubs:
    r=simulate_cascade(capacity,load,neighbors,[hub],tolerance=1.0,max_rounds=20,check_initial=True)
    assert r["converged"],f"Simulation did not converge for {hub}"
    assert r["conserved"],f"Load was not conserved for {hub}"
    real_rows.append({"initial_failed_hub":hub,"initial_failures":1,"total_hubs_failed":r["n_failed"],
        "secondary_hubs_failed":r["n_failed"]-1,"secondary_cascade":r["n_failed"]>1,"rounds":r["n_rounds"],
        "unserved_load_legs_per_day":r["unserved_load"],"unserved_share":r["unserved_share"],
        "max_alive_utilization":r["max_utilization"],"converged":r["converged"],"conserved":r["conserved"],
        "failed_hubs":";".join(r["failed"])})
real_results=pd.DataFrame(real_rows).sort_values(["total_hubs_failed","unserved_share","initial_failed_hub"],ascending=[False,False,True]).reset_index(drop=True)
assert len(real_results)==100 and real_results.initial_failed_hub.is_unique
assert real_results.converged.all() and real_results.conserved.all()
critical=real_results.loc[real_results.secondary_cascade].copy().reset_index(drop=True)
secondary_count=int(real_results.secondary_cascade.sum())
largest_cascade=int(real_results.total_hubs_failed.max())
print("Initial failures:",len(real_results),"| causing secondary cascades:",secondary_count,"| largest cascade including initial hub:",largest_cascade)
display(real_results.head(10))

Initial failures: 100 | causing secondary cascades: 1 | largest cascade including initial hub: 4


,initial_failed_hub,initial_failures,total_hubs_failed,secondary_hubs_failed,secondary_cascade,rounds,unserved_load_legs_per_day,unserved_share,max_alive_utilization,converged,conserved,failed_hubs
0,IND000000ACA,1,4,3,True,3,4.34,0.011342,1.0,True,True,IND000000ACA;IND143001AAA;IND144001AAB;IND1600...
1,IND000000AAL,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND000000AAL
2,IND000000ACB,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND000000ACB
3,IND000000ACT,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND000000ACT
4,IND000000AEL,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND000000AEL
5,IND000000AFT,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND000000AFT
6,IND110030AAD,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND110030AAD
7,IND110037AAK,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND110037AAK
8,IND110037AAM,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND110037AAM
9,IND110044AAB,1,1,0,False,1,0.00,0.000000,1.0,True,True,IND110044AAB


## Critical hubs and cascade-size inspection

Only initial failures with secondary failures appear in the critical-hub table. If none exist, the saved file will contain headers and zero rows. These are structural sensitivity results, not measured hub criticality.

In [8]:
display(critical)

,initial_failed_hub,initial_failures,total_hubs_failed,secondary_hubs_failed,secondary_cascade,rounds,unserved_load_legs_per_day,unserved_share,max_alive_utilization,converged,conserved,failed_hubs
0,IND000000ACA,1,4,3,True,3,4.34,0.011342,1.0,True,True,IND000000ACA;IND143001AAA;IND144001AAB;IND1600...


## Limitations and interpretation

- The directed core lanes become an undirected neighbor graph.
- The simulator redistributes accumulated scalar load; it does not preserve OD flows or reroute only excess traffic.
- Dispatch capacity is used as structural spare room; receiving capacity is unmeasured.
- Destination-only zero-dispatch hubs, physical storage, warehouse throughput and operational queues are not represented.
- Synthetic experiments are sensitivity demonstrations, not observed traffic.
- Cascade counts depend on this 14-day mean baseline, capacity recipe, topology, tolerance and redistribution rule. They are not probabilities or forecasts.
- Directed OD rerouting and queue behavior belong to the separate optimizer and daily queue simulator.

## Fresh-data conclusion

The conclusion below is generated from the current run’s `real_results`, not retained notebook outputs.

In [9]:
print(f"Analyzed {len(core_hubs)} hubs and {len(edge_pairs)} frozen directed lanes.")
print("Baseline load: mean daily outgoing departures per hub, Sep 12–25 2018 inclusive, including zero days.")
print("Capacity: ceil of each hub's linearly interpolated 95th percentile over those 14 daily outgoing counts.")
print(f"Initial hub failures tested: {len(real_results)}; scenarios with secondary cascades: {secondary_count}.")
print(f"Largest cascade: {largest_cascade} total failed hubs including the initial failure.")
if len(critical): print("Critical initial hubs:",", ".join(critical.head(5).initial_failed_hub.astype(str)))
else: print("No tested single-hub failure caused a secondary cascade under this structural setup.")
print("This result is a structural scalar-load stress sensitivity, not operational OD rerouting or physical capacity validation.")

Analyzed 100 hubs and 292 frozen directed lanes.
Baseline load: mean daily outgoing departures per hub, Sep 12–25 2018 inclusive, including zero days.
Capacity: ceil of each hub's linearly interpolated 95th percentile over those 14 daily outgoing counts.
Initial hub failures tested: 100; scenarios with secondary cascades: 1.
Largest cascade: 4 total failed hubs including the initial failure.
Critical initial hubs: IND000000ACA
This result is a structural scalar-load stress sensitivity, not operational OD rerouting or physical capacity validation.


## Save versioned outputs and provenance

The output folder is selected only if unused. Files are created exclusively; a rerun selects the next `final_vN` directory. The success marker is created last and only after read-back checks pass.

In [10]:
output_frames={"real_data_cascade_results_v1.csv":real_results,"critical_hub_results_v1.csv":critical,"synthetic_stress_results_v1.csv":synthetic_stress}
plot_path=OUT_DIR/"cascade_stress_curve_v1.png"; log_path=OUT_DIR/"execution_log_v1.txt"
manifest_path=OUT_DIR/"run_manifest_v1.json"; success_path=OUT_DIR/"_SUCCESS_v1.json"
all_paths=[OUT_DIR/n for n in output_frames]+[plot_path,log_path,manifest_path,success_path]
assert not any(p.exists() for p in all_paths),"Refusing to overwrite existing output"
OUT_DIR.mkdir(parents=False,exist_ok=False)
for name,frame in output_frames.items():
    with (OUT_DIR/name).open("x",encoding="utf-8",newline="") as f: frame.to_csv(f,index=False)
with plot_path.open("xb") as f: fig.savefig(f,format="png",dpi=160,bbox_inches="tight")
plt.close(fig)
log=(f"Status: all validations passed\nNotebook: {NOTEBOOK_PATH}\nPython={platform.python_version()} pandas={pd.__version__} numpy={np.__version__} matplotlib={matplotlib.__version__}\n"
     f"Hubs={len(core_hubs)} directed_lanes={len(edge_pairs)} period=2018-09-12..2018-09-25 inclusive\n"
     f"departures={int(hub_daily.outgoing_legs.sum())} zero_hub_days={int((hub_daily.outgoing_legs==0).sum())}\n"
     f"initial_failures={len(real_results)} secondary_cascades={secondary_count} largest_cascade={largest_cascade}\n")
with log_path.open("x",encoding="utf-8",newline="") as f: f.write(log)
def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
saved={name:pd.read_csv(OUT_DIR/name) for name in output_frames}
assert len(saved["real_data_cascade_results_v1.csv"])==100
assert len(saved["critical_hub_results_v1.csv"])==secondary_count
assert len(saved["synthetic_stress_results_v1.csv"])==44
assert saved["real_data_cascade_results_v1.csv"].initial_failed_hub.is_unique
assert saved["real_data_cascade_results_v1.csv"].converged.all() and saved["real_data_cascade_results_v1.csv"].conserved.all()
inputs=[DATA/"core_nodes.csv",DATA/"core_edges.csv",DATA/"core_train_legs.csv",DATA/"hub_daily_departures.csv",DATA/"split_trip_ids.csv",ROOT/"sim/simulate.py",ROOT/"docs/assumptions.md",ROOT/"docs/cascade_rules_v1.md"]
checks={"split_unique":True,"core_rows_master_train":True,"lane_keys_valid_and_unique":True,"in_window_lanes_frozen_core":True,"daily_grid_complete_and_finite":True,"zero_days_retained":True,"independent_daily_aggregate_matches":True,"capacity_recipe_reproduced":True,"all_real_cascades_converged_and_conserved":True,"saved_outputs_read_back_and_validated":True}
manifest={"status":"complete","notebook":{"path":NOTEBOOK_PATH,"version":"final_v3"},"execution_utc":datetime.now(timezone.utc).isoformat(),
 "environment":{"python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,"matplotlib":matplotlib.__version__,"networkx":importlib.metadata.version("networkx")},
 "inputs":{p.relative_to(ROOT).as_posix():sha256(p) for p in inputs},
 "outputs":{(OUT_DIR/n).relative_to(ROOT).as_posix():sha256(OUT_DIR/n) for n in output_frames},
 "figure":{"path":plot_path.relative_to(ROOT).as_posix(),"sha256":sha256(plot_path)},"execution_log":{"path":log_path.relative_to(ROOT).as_posix(),"sha256":sha256(log_path)},
 "analysis":{"hubs":len(core_hubs),"directed_lanes":len(edge_pairs),"baseline_period":["2018-09-12","2018-09-25"],"baseline_days":14,
 "baseline_definition":"mean daily outgoing departures per hub, including zero-departure days","departures_in_window":int(hub_daily.outgoing_legs.sum()),
 "zero_departure_hub_days":int((hub_daily.outgoing_legs==0).sum()),"capacity_definition":"ceil of per-hub linear 95th percentile of 14 daily outgoing counts",
 "initial_hub_failures_tested":len(real_results),"secondary_cascade_scenarios":secondary_count,"largest_cascade_including_initial":largest_cascade},
 "validation_checks":checks,"limitations":["undirected graph derived from directed lanes","scalar load; no OD preservation","whole accumulated load is redistributed, not only excess",
 "dispatch proxy is not measured receiving capacity","not operational rerouting, warehouse capacity, or queue validation"]}
with manifest_path.open("x",encoding="utf-8",newline="") as f: json.dump(manifest,f,indent=2)
success={"status":"complete","manifest":manifest_path.relative_to(ROOT).as_posix(),"manifest_sha256":sha256(manifest_path),
 "real_results_rows":len(saved["real_data_cascade_results_v1.csv"]),"critical_hub_rows":len(saved["critical_hub_results_v1.csv"]),"synthetic_stress_rows":len(saved["synthetic_stress_results_v1.csv"])}
with success_path.open("x",encoding="utf-8",newline="") as f: json.dump(success,f,indent=2)
print("Saved outputs:",OUT_DIR.relative_to(ROOT))
for p in all_paths: print(p.relative_to(ROOT),sha256(p))
print("Success marker created last.")

Saved outputs: experiments\day4\final_v2
experiments\day4\final_v2\real_data_cascade_results_v1.csv 6b0ccb1ece81187af1b03ab149632c29e194d356320389a32e0f5f3d6b8bc749
experiments\day4\final_v2\critical_hub_results_v1.csv bf8f08fea4151dbb8d89298aeb4678a5b2360b7136de7cf8ca3135eeaf19d429
experiments\day4\final_v2\synthetic_stress_results_v1.csv d830167e08fb5cb147162b296873ea157754f912bea511bd2170a11aa519e025
experiments\day4\final_v2\cascade_stress_curve_v1.png 4c2d17b80543d8452b453339dbc21e71e883b8b1eaa42e5cb44d419f77b7166c
experiments\day4\final_v2\execution_log_v1.txt bc1477bd3c3e1da12cbdf09d64c7db929a54527774e55a2ad0f18761494329ba
experiments\day4\final_v2\run_manifest_v1.json 0d280c98e5cd9d18b0c84ff9fa80659df32d0bcd735ac911b7ed007dfa06f266
experiments\day4\final_v2\_SUCCESS_v1.json 483985e78e8f8c38d6aad7a62841c52457d84b0d17f5f2901149be43a9a7dac7
Success marker created last.
